# E-commerce Recommendation Engine - Data Exploration and Cleaning

**Objective:** Explore and clean the UCI Online Retail dataset (real UK online retailer
transactions, Dec 2010 to Dec 2011) and prepare a customer-item interaction table for
recommendation modelling.

**Source:** UCI Machine Learning Repository, "Online Retail" dataset
(https://archive.ics.uci.edu/dataset/352/online+retail). 541,909 transaction line items.

**Output:** `clean_transactions.csv` - cleaned, de-duplicated purchase records with one
row per (invoice, product) line, ready for building a user-item interaction matrix.

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.width', 120)
plt.style.use('seaborn-v0_8-whitegrid')

print("Setup complete.")

Setup complete.


## 2. Load Raw Data

In [2]:
df = pd.read_csv("../data/online_retail_2010_2011.csv.gz", encoding="latin1",
                 compression="gzip", dtype={'CustomerID': 'Int64'})
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], format="%m/%d/%Y %H:%M")

print("Raw shape:", df.shape)
df.head()

Raw shape: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom


In [3]:
print("Date range:", df['InvoiceDate'].min(), "to", df['InvoiceDate'].max())
print("Unique invoices:", df['InvoiceNo'].nunique())
print("Unique products:", df['StockCode'].nunique())
print("Unique customers (incl. missing):", df['CustomerID'].nunique())
print("Unique countries:", df['Country'].nunique())
print()
print("Missing values:")
print(df.isnull().sum())

Date range: 2010-12-01 08:26:00 to 2011-12-09 12:50:00
Unique invoices: 25900
Unique products: 4070
Unique customers (incl. missing): 4372
Unique countries: 38

Missing values:


InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


## 3. Data Quality Issues

Three known issues in this dataset need handling before it can be used for recommendations:

1. **Cancellations** - `InvoiceNo` values starting with 'C' represent returns/cancellations,
   not purchases.
2. **Missing CustomerID** - about a quarter of rows have no customer identifier attached
   (likely guest or POS transactions) and cannot be tied to a specific shopper.
3. **Non-product stock codes** - codes like `POST` (postage) and `M` (manual adjustment)
   are administrative line items, not purchasable products.

In [4]:
is_cancel = df['InvoiceNo'].astype(str).str.startswith('C')
print("Cancellation rows:", is_cancel.sum())
print("Missing CustomerID rows:", df['CustomerID'].isna().sum())
print("Negative quantity rows:", (df['Quantity'] < 0).sum())
print("Zero/negative price rows:", (df['UnitPrice'] <= 0).sum())

Cancellation rows: 9288
Missing CustomerID rows: 135080
Negative quantity rows: 10624
Zero/negative price rows: 2517


## 4. Clean the Data

In [5]:
non_product_codes = {'POST', 'D', 'M', 'BANK CHARGES', 'PADS', 'DOT', 'CRUK'}

clean = df[
    (~is_cancel)
    & (df['CustomerID'].notna())
    & (df['Quantity'] > 0)
    & (df['UnitPrice'] > 0)
    & (~df['StockCode'].isin(non_product_codes))
].copy()

clean['CustomerID'] = clean['CustomerID'].astype(int)
clean['StockCode'] = clean['StockCode'].astype(str).str.strip()
clean['TotalPrice'] = clean['Quantity'] * clean['UnitPrice']

print(f"Rows retained: {len(clean):,} ({len(clean)/len(df)*100:.2f}% of raw)")
print("Unique customers:", clean['CustomerID'].nunique())
print("Unique products:", clean['StockCode'].nunique())
print("Unique invoices:", clean['InvoiceNo'].nunique())
print("Total revenue: £{:,.2f}".format(clean['TotalPrice'].sum()))

Rows retained: 396,470 (73.16% of raw)
Unique customers: 4334
Unique products: 3660
Unique invoices: 18405
Total revenue: £8,767,752.65


## 5. Customer Purchase Behaviour

In [6]:
orders_per_customer = clean.groupby('CustomerID')['InvoiceNo'].nunique()

print(f"Orders per customer: mean={orders_per_customer.mean():.2f}, "
      f"median={orders_per_customer.median():.1f}, max={orders_per_customer.max()}")
print(f"Customers with only 1 order: {(orders_per_customer == 1).sum()} "
      f"({(orders_per_customer==1).mean()*100:.2f}%)")
print(f"Customers with 2+ orders: {(orders_per_customer >= 2).sum()} "
      f"({(orders_per_customer>=2).mean()*100:.2f}%)")

fig, ax = plt.subplots(figsize=(8, 4))
orders_per_customer.clip(upper=20).plot(kind='hist', bins=20, ax=ax, color='#4C72B0')
ax.set_xlabel('Orders per customer (clipped at 20)')
ax.set_ylabel('Number of customers')
ax.set_title('Distribution of Orders per Customer')
plt.tight_layout()
plt.savefig('orders_per_customer.png', dpi=100)
plt.show()

Orders per customer: mean=4.25, median=2.0, max=206
Customers with only 1 order: 1505 (34.73%)
Customers with 2+ orders: 2829 (65.27%)


<Figure size 800x400 with 1 Axes>

Repeat purchase behaviour is the key requirement for a recommender system: with 65%+ of
customers placing two or more orders, there is enough purchase history per customer for
collaborative filtering to find meaningful patterns, unlike single-purchase-dominated
datasets where every customer is effectively a cold start.

## 6. Product Popularity

In [7]:
items_per_product = clean.groupby('StockCode')['CustomerID'].nunique().sort_values(ascending=False)
desc_map = clean.groupby('StockCode')['Description'].first()

top10 = items_per_product.head(10)
print("Top 10 products by number of distinct customers who bought them:")
for code, n in top10.items():
    print(f"  {code}: {desc_map[code].strip():<45} n_customers={n}")

Top 10 products by number of distinct customers who bought them:
  22423: REGENCY CAKESTAND 3 TIER                      n_customers=881
  85123A: WHITE HANGING HEART T-LIGHT HOLDER            n_customers=856
  47566: PARTY BUNTING                                 n_customers=708
  84879: ASSORTED COLOUR BIRD ORNAMENT                 n_customers=678
  22720: SET OF 3 CAKE TINS PANTRY DESIGN              n_customers=640
  21212: PACK OF 72 RETROSPOT CAKE CASES               n_customers=635
  85099B: JUMBO BAG RED RETROSPOT                       n_customers=635
  22086: PAPER CHAIN KIT 50'S CHRISTMAS                n_customers=613
  22457: NATURAL SLATE HEART CHALKBOARD                n_customers=587
  22138: BAKING SET 9 PIECE RETROSPOT                  n_customers=581


## 7. Interaction Matrix Sparsity

In [8]:
n_users = clean['CustomerID'].nunique()
n_items = clean['StockCode'].nunique()
n_interactions = clean.groupby(['CustomerID', 'StockCode']).ngroups
density = n_interactions / (n_users * n_items) * 100

print(f"User-item matrix: {n_users:,} users x {n_items:,} items = {n_users*n_items:,} cells")
print(f"Observed interactions: {n_interactions:,} ({density:.4f}% density)")

User-item matrix: 4,334 users x 3,660 items = 15,862,440 cells
Observed interactions: 266,250 (1.6785% density)


At 1.68% density, the matrix is sparse but well within the range where collaborative
filtering is effective — dense enough that items share overlapping buyers, sparse enough
that a naive popularity ranking leaves clear room for personalization to add value.

## 8. Save Cleaned Data

In [9]:
clean.to_csv("clean_transactions.csv", index=False)
print("Saved clean_transactions.csv:", clean.shape)

Saved clean_transactions.csv: (396470, 9)
